# Internet Of Things: second challenge part two
 
## Authors
- Lorenzo Bardelli: 
- Pietro Pizzoccheri (Team Leader): 10797420

## Part 2: Exercise (EQ1 & EQ2)

### EQ1: Energy Consumption
- Logic: We calculate total communication energy in microjoules from TX/RX active bits over 1 hour.
- CoAP (direct): Sensor sends CON PUT to actuator, actuator sends ACK, then both sleep.
- MQTT (via gateway): Sensor and actuator reconnect on wake-ups; sensor publishes with QoS 1, actuator acknowledges. A SUBSCRIBE/SUBACK overhead is included once per hour.
- EQ1a Answer: `2203.2 μJ`
- EQ1b Answer: `7563.52 μJ`

### EQ2: Protocol for Long-Term Operation (Actuator sleeps 30 minutes)
- Logic: Sensor samples every 2 minutes; direct CoAP would waste battery on repeated retries while actuator sleeps.
- Answer: MQTT is preferable using `retain = true` and persistent session behavior, so the actuator receives only the latest value when it wakes.
- Suggested final sentence (short form): MQTT with retained publish minimizes battery usage by avoiding repeated CoAP retransmissions during actuator sleep and delivering only the newest value at wake-up.

In [ ]:
# EQ1 computation
ETX_nJ = 50
ERX_nJ = 58
events_per_hour = 60 // 2  # one measurement every 2 minutes -> 30 events/hour

def bits(n_bytes: int) -> int:
    return 8 * n_bytes

# CoAP direct per event: sensor TX PUT + sensor RX ACK + actuator RX PUT + actuator TX ACK
coap_per_event_nJ = (
    bits(70) * ETX_nJ +
    bits(15) * ERX_nJ +
    bits(70) * ERX_nJ +
    bits(15) * ETX_nJ
)
eq1a_uJ = coap_per_event_nJ * events_per_hour / 1000.0

# MQTT via gateway per event (sensor + actuator side, QoS1)
sensor_per_event_nJ = bits(60 + 75 + 60) * ETX_nJ + bits(50 + 50) * ERX_nJ
actuator_per_event_nJ = bits(60 + 50 + 60) * ETX_nJ + bits(50 + 75) * ERX_nJ

# sbscription overhead once per hour
subscribe_once_nJ = bits(65) * ETX_nJ + bits(55) * ERX_nJ

eq1b_uJ = (events_per_hour * (sensor_per_event_nJ + actuator_per_event_nJ) + subscribe_once_nJ) / 1000.0

print(f"EQ1a (CoAP): {eq1a_uJ:.1f} μJ")
print(f"EQ1b (MQTT): {eq1b_uJ:.2f} μJ")
print("Matches report values:", round(eq1a_uJ, 1) == 2203.2 and round(eq1b_uJ, 2) == 7563.52)

In [ ]:
# EQ2 
sensor_period_min = 2
actuator_wakeup_min = 30
samples_while_actuator_sleeping = actuator_wakeup_min // sensor_period_min

print(f"Sensor samples during one actuator sleep window: {samples_while_actuator_sleeping}")

choice = "MQTT"
required_parameter = "retain = true"
optimized_metric = "battery lifetime (energy efficiency)"

print(f"Recommended protocol: {choice}")
print(f"Required parameter: {required_parameter}")
print(f"Optimized metric: {optimized_metric}")
print(
    "Reason: with long actuator sleep, CoAP direct causes repeated failed/retransmitted attempts; "
    "MQTT with retained publish lets actuator receive only the latest value on wake-up."
)

### Part 2 (EQs)
| Question | Final Answer |
|---|---|
| EQ1a | 2203.2 μJ |
| EQ1b | 7563.52 μJ |
| EQ2 | MQTT with retain=true and persistent session behavior |